## Key Takeaways

**What You've Learned:**
- ✅ **Image Preprocessing**: Normalized pixel values and reshaped images for CNN input
- ✅ **CNN Architecture**: Built a model with convolutional layers for feature extraction
- ✅ **Model Training**: Trained and validated the model on MNIST dataset
- ✅ **Performance Evaluation**: Analyzed accuracy, precision, recall, and confusion matrices
- ✅ **Visualization**: Created comprehensive plots for results interpretation
- ✅ **Real-world Application**: Made predictions on new digit images

**Model Performance:**
- Typical test accuracy: > 98%
- Strong generalization from training to test data
- Very few misclassified samples

**Future Enhancements:**
- Implement data augmentation for improved robustness
- Experiment with deeper architectures (ResNet, DenseNet)
- Apply transfer learning with pre-trained models
- Deploy as a web service or mobile app
- Test on different handwriting datasets (SVHN, USPS)

In [ ]:
# Show misclassified examples
def show_misclassified(num_samples=12):
    """Display examples where the model made wrong predictions"""
    
    # Find misclassified indices
    misclassified_mask = y_pred != y_test
    misclassified_indices = np.where(misclassified_mask)[0]
    
    if len(misclassified_indices) == 0:
        print("Perfect predictions! No misclassified samples found.")
        return
    
    # Select random misclassified samples
    selected_indices = np.random.choice(misclassified_indices, 
                                       min(num_samples, len(misclassified_indices)), 
                                       replace=False)
    
    # Calculate grid size
    n_cols = 6
    n_rows = (len(selected_indices) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 2*n_rows))
    fig.suptitle('Misclassified Samples', fontsize=14, fontweight='bold', color='red')
    axes = axes.flatten()
    
    for idx, sample_idx in enumerate(selected_indices):
        ax = axes[idx]
        true_label = y_test[sample_idx]
        pred_label = y_pred[sample_idx]
        confidence = y_pred_prob[sample_idx][pred_label]
        
        ax.imshow(x_test[sample_idx], cmap='gray')
        ax.set_title(f'True: {true_label}, Pred: {pred_label}\nConf: {confidence:.2%}',
                    fontsize=10, color='red', fontweight='bold')
        ax.axis('off')
    
    # Hide empty subplots
    for idx in range(len(selected_indices), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

print(f"Total misclassified samples: {np.sum(y_pred != y_test)} out of {len(y_test)}")
print(f"Misclassification rate: {np.mean(y_pred != y_test)*100:.2f}%\n")

show_misclassified(num_samples=12)


In [ ]:
# Analyze per-digit accuracy
per_digit_accuracy = {}
for digit in range(10):
    # Get indices of true labels equal to this digit
    digit_mask = y_test == digit
    digit_predictions = y_pred[digit_mask]
    digit_true_labels = y_test[digit_mask]
    
    # Calculate accuracy for this digit
    accuracy = np.mean(digit_predictions == digit_true_labels)
    per_digit_accuracy[digit] = accuracy

# Plot per-digit accuracy
digits = list(per_digit_accuracy.keys())
accuracies = list(per_digit_accuracy.values())

plt.figure(figsize=(12, 5))
bars = plt.bar(digits, accuracies, color='steelblue', edgecolor='black', linewidth=1.5)

# Color bars based on accuracy
for bar, acc in zip(bars, accuracies):
    if acc >= 0.99:
        bar.set_color('green')
    elif acc >= 0.95:
        bar.set_color('steelblue')
    else:
        bar.set_color('orange')

plt.xlabel('Digit', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Per-Digit Classification Accuracy on Test Set', fontsize=14, fontweight='bold')
plt.xticks(digits)
plt.ylim([0.85, 1.02])
plt.grid(axis='y', alpha=0.3)

# Add accuracy values on top of bars
for digit, accuracy in per_digit_accuracy.items():
    plt.text(digit, accuracy + 0.005, f'{accuracy:.2%}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("\nPer-Digit Accuracy:")
for digit, accuracy in per_digit_accuracy.items():
    print(f"Digit {digit}: {accuracy:.4f} ({accuracy*100:.2f}%)")


In [ ]:
# Plot training history - Accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Model Training History', fontsize=14, fontweight='bold')

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].set_title('Model Accuracy Over Epochs', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Loss', fontsize=11)
axes[1].set_title('Model Loss Over Epochs', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print training summary
print("Training Summary:")
print(f"Final Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")
print(f"Final Training Loss: {history.history['loss'][-1]:.4f}")
print(f"Final Validation Loss: {history.history['val_loss'][-1]:.4f}")


## Section 9: Visualize Results

Create visualizations including sample predictions, confusion matrices, training history plots, and accuracy/loss curves.

In [ ]:
# Make predictions on random test samples
def predict_and_display(num_samples=10):
    """Make predictions on random samples and display results"""
    
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    fig.suptitle('Predictions on Test Samples', fontsize=14, fontweight='bold')
    
    # Select random indices
    random_indices = np.random.choice(len(x_test), num_samples, replace=False)
    
    for idx, ax in enumerate(axes.flat):
        sample_idx = random_indices[idx]
        
        # Get prediction
        sample_image = x_test_reshaped[sample_idx:sample_idx+1]
        prediction = model.predict(sample_image, verbose=0)
        predicted_label = np.argmax(prediction)
        confidence = prediction[0][predicted_label]
        true_label = y_test[sample_idx]
        
        # Display image
        ax.imshow(x_test[sample_idx], cmap='gray')
        
        # Color based on correctness
        color = 'green' if predicted_label == true_label else 'red'
        ax.set_title(f'True: {true_label}, Pred: {predicted_label}\nConfidence: {confidence:.2%}',
                    fontsize=10, color=color, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Display predictions
predict_and_display(num_samples=10)


## Section 8: Make Predictions on New Digits

Use the trained model to make predictions on new handwritten digit images and display the predicted labels with confidence scores.

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(10), yticklabels=range(10),
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - MNIST Digit Classification', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Evaluate on test set
print("Evaluating model on test dataset...\n")
test_loss, test_accuracy = model.evaluate(x_test_reshaped, y_test_categorical, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Make predictions
y_pred_prob = model.predict(x_test_reshaped, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

# Calculate metrics
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"\nPrecision (Weighted): {precision:.4f}")
print(f"Recall (Weighted): {recall:.4f}")

# Classification Report
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=[str(i) for i in range(10)]))


## Section 7: Evaluate Model Performance

Evaluate the trained model on the test dataset, calculate accuracy, precision, recall, and generate a confusion matrix.

In [ ]:
# Train the model
print("Training the CNN model...")
print("This may take a few minutes...\n")

history = model.fit(
    x_train_reshaped, y_train_categorical,
    epochs=15,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

print("\nTraining completed!")


## Section 6: Train the Model

Train the CNN on the MNIST training dataset with appropriate epochs and batch sizes, monitoring training and validation metrics.

In [ ]:
# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")
print(f"Optimizer: Adam (learning rate=0.001)")
print(f"Loss Function: Categorical Crossentropy")
print(f"Metrics: Accuracy")


## Section 5: Compile the Model

Compile the model by specifying the optimizer, loss function, and evaluation metrics.

In [ ]:
# Build CNN model
model = models.Sequential([
    # First Convolutional Block
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Second Convolutional Block
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Third Convolutional Block
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    # Flatten and Dense Layers
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    
    # Output Layer
    layers.Dense(10, activation='softmax')
])

# Display model architecture
print("CNN Model Architecture:")
model.summary()


## Section 4: Build the CNN Model

Design a CNN architecture with convolutional layers, pooling layers, and dense layers to extract features and classify digits.

In [ ]:
# Normalize pixel values to [0, 1]
x_train_normalized = x_train.astype('float32') / 255.0
x_test_normalized = x_test.astype('float32') / 255.0

# Reshape images: add channel dimension for CNN (28, 28, 1)
x_train_reshaped = x_train_normalized.reshape(-1, 28, 28, 1)
x_test_reshaped = x_test_normalized.reshape(-1, 28, 28, 1)

# Convert labels to one-hot encoding
y_train_categorical = to_categorical(y_train, 10)
y_test_categorical = to_categorical(y_test, 10)

print("Preprocessed Data Information:")
print(f"Training data shape: {x_train_reshaped.shape}")
print(f"Test data shape: {x_test_reshaped.shape}")
print(f"Training labels shape: {y_train_categorical.shape}")
print(f"Test labels shape: {y_test_categorical.shape}")
print(f"\nNormalized pixel value range: [{x_train_normalized.min()}, {x_train_normalized.max()}]")
print(f"\nExample one-hot encoded label: {y_train_categorical[0]}")


## Section 3: Preprocess the Data

Normalize pixel values to the range [0, 1], reshape images, and split data into training and testing sets.

In [ ]:
# Visualize sample digits from the MNIST dataset
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sample Handwritten Digits from MNIST Dataset', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    idx = np.random.randint(0, len(x_train))
    ax.imshow(x_train[idx], cmap='gray')
    ax.set_title(f'Label: {y_train[idx]}', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

# Display digit distribution
print("\nDigit Distribution in Training Set:")
unique, counts = np.unique(y_train, return_counts=True)
for digit, count in zip(unique, counts):
    print(f"Digit {digit}: {count} samples")


In [ ]:
# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print("Dataset Information:")
print(f"Training data shape: {x_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test data shape: {x_test.shape}")
print(f"Test labels shape: {y_test.shape}")
print(f"\nUnique digits in training set: {np.unique(y_train)}")
print(f"Pixel value range: [{x_train.min()}, {x_train.max()}]")


## Section 2: Load and Explore the MNIST Dataset

Load the MNIST dataset from Keras, examine its structure, shape, and visualize sample digits with their labels.

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)
print("NumPy version:", np.__version__)

## Section 1: Import Required Libraries

Import necessary libraries including TensorFlow, Keras, NumPy, Matplotlib, and scikit-learn for building and training the CNN model.

# Handwritten Digit Recognizer - CNN with MNIST Dataset

Train a Convolutional Neural Network (CNN) to recognize handwritten digits using the MNIST dataset. This notebook demonstrates image processing, neural network design, and deep learning fundamentals.

**Topics Covered:**
- Image preprocessing and normalization
- Convolutional Neural Network architecture
- Model training and validation
- Performance evaluation with metrics
- Prediction and visualization